# 🌍 Luftqualität & Gesundheit – Starter Notebook
**Data Pipeline: UBA + EEA → PostgreSQL**

Dieses Notebook lädt Luftqualitätsdaten aus zwei Quellen,
bereinigt sie und speichert sie in einer PostgreSQL-Datenbank.

---
**Quellen:**
- 🇩🇪 Umweltbundesamt (UBA) – Deutschland, stündlich
- 🇪🇺 European Environment Agency (EEA) – Europa, validiert

**Stack:** Python · Pandas · SQLAlchemy · PostgreSQL · airbase

## 0. Setup & Installation

In [ ]:
# Pakete installieren (einmalig)
# !pip install pandas sqlalchemy psycopg2-binary airbase requests python-dotenv

In [ ]:
import pandas as pd
import requests
import json
import os
from datetime import datetime, timedelta
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

print('✅ Imports erfolgreich')

## 1. Datenbankverbindung (PostgreSQL)

Verbindungsstring anpassen – lokal oder z.B. Neon.tech (kostenlos für Teams)

In [ ]:
# --- Verbindung konfigurieren ---
DB_CONFIG = {
    'host':     'localhost',       # oder neon.tech host
    'port':     5432,
    'database': 'luftqualitaet',
    'user':     'postgres',
    'password': 'dein_passwort'    # besser: aus .env laden
}

CONNECTION_STRING = (
    f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

engine = create_engine(CONNECTION_STRING)

# Verbindung testen
with engine.connect() as conn:
    result = conn.execute(text('SELECT version()'))
    print('✅ DB-Verbindung erfolgreich:', result.fetchone()[0][:40])

## 2. Datenbankschema erstellen

In [ ]:
schema_sql = """
-- Messstationen
CREATE TABLE IF NOT EXISTS stationen (
    station_id      VARCHAR(50) PRIMARY KEY,
    station_name    VARCHAR(200),
    bundesland      VARCHAR(100),
    land            VARCHAR(10),
    station_typ     VARCHAR(50),   -- z.B. 'urban', 'suburban', 'rural'
    latitude        FLOAT,
    longitude       FLOAT,
    quelle          VARCHAR(10)    -- 'UBA' oder 'EEA'
);

-- Messdaten
CREATE TABLE IF NOT EXISTS messdaten (
    id              SERIAL PRIMARY KEY,
    station_id      VARCHAR(50) REFERENCES stationen(station_id),
    zeitpunkt       TIMESTAMP NOT NULL,
    schadstoff      VARCHAR(20) NOT NULL,  -- 'PM10', 'PM2.5', 'NO2', 'O3'
    wert            FLOAT,
    einheit         VARCHAR(20) DEFAULT 'µg/m³',
    validiert       BOOLEAN DEFAULT FALSE,
    quelle          VARCHAR(10)
);

-- Index für schnelle Zeitreihenabfragen
CREATE INDEX IF NOT EXISTS idx_messdaten_zeitpunkt 
    ON messdaten(station_id, schadstoff, zeitpunkt);

-- Jahresbilanzen (aggregiert)
CREATE TABLE IF NOT EXISTS jahresbilanzen (
    id              SERIAL PRIMARY KEY,
    station_id      VARCHAR(50),
    jahr            INT,
    schadstoff      VARCHAR(20),
    jahresmittel    FLOAT,
    max_tageswert   FLOAT,
    ueberschreitungen INT,          -- Tage über Grenzwert
    quelle          VARCHAR(10),
    UNIQUE(station_id, jahr, schadstoff)
);
"""

with engine.connect() as conn:
    conn.execute(text(schema_sql))
    conn.commit()

print('✅ Schema erstellt (Tabellen: stationen, messdaten, jahresbilanzen)')

## 3. Quelle 1: UBA API

Stundendaten für Deutschland – PM10, PM2.5, NO₂, Ozon

Dokumentation: https://www.umweltbundesamt.de/daten/luft/luftdaten

In [ ]:
def uba_stationen_laden():
    """Lädt alle UBA-Messstationen mit Metadaten."""
    url = 'https://luftdaten.umweltbundesamt.de/api/stations'
    
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        stationen = []
        for station in data:
            stationen.append({
                'station_id':   'UBA_' + str(station.get('id', '')),
                'station_name': station.get('name', ''),
                'bundesland':   station.get('state', ''),
                'land':         'DE',
                'station_typ':  station.get('type', ''),
                'latitude':     station.get('lat'),
                'longitude':    station.get('lon'),
                'quelle':       'UBA'
            })
        
        df = pd.DataFrame(stationen)
        print(f'✅ {len(df)} UBA-Stationen geladen')
        return df
    
    except Exception as e:
        print(f'⚠️  UBA Stationen API Fehler: {e}')
        print('   → Nutze stattdessen das Webportal: umweltbundesamt.de/daten/luft/luftdaten')
        return pd.DataFrame()


def uba_messdaten_laden(datum_von='2023-01-01', datum_bis='2023-12-31',
                         schadstoffe=['PM10', 'PM2.5', 'NO2', 'O3']):
    """
    Lädt Messdaten vom UBA API.
    Schadstoff-Codes: 1=PM10, 3=O3, 5=NO2, 9=PM2.5
    """
    schadstoff_codes = {'PM10': 1, 'O3': 3, 'NO2': 5, 'PM2.5': 9}
    base_url = 'https://luftdaten.umweltbundesamt.de/api/data/air'
    
    alle_daten = []
    
    for schadstoff in schadstoffe:
        code = schadstoff_codes.get(schadstoff)
        if not code:
            continue
            
        params = {
            'date_from': datum_von,
            'date_to':   datum_bis,
            'time_from': '1',
            'time_to':   '24',
            'component': code,
            'scope':     2   # Tagesmittelwerte
        }
        
        try:
            r = requests.get(base_url, params=params, timeout=60)
            r.raise_for_status()
            data = r.json()
            
            # Daten aus verschachteltem JSON extrahieren
            if 'data' in data:
                for station_id, zeitreihe in data['data'].items():
                    for zeitpunkt, wert in zeitreihe.items():
                        if wert is not None:
                            alle_daten.append({
                                'station_id': 'UBA_' + station_id,
                                'zeitpunkt':  pd.to_datetime(zeitpunkt),
                                'schadstoff': schadstoff,
                                'wert':       float(wert[0]) if isinstance(wert, list) else float(wert),
                                'validiert':  True,
                                'quelle':     'UBA'
                            })
            
            print(f'  ✅ {schadstoff}: Daten geladen')
            
        except Exception as e:
            print(f'  ⚠️  {schadstoff} Fehler: {e}')
    
    df = pd.DataFrame(alle_daten)
    print(f'\n✅ UBA gesamt: {len(df):,} Messwerte')
    return df


# --- Ausführen ---
print('📡 Lade UBA-Daten (2022, Tagesmittelwerte)...')
uba_messdaten = uba_messdaten_laden(
    datum_von='2022-01-01',
    datum_bis='2022-12-31'
)
uba_messdaten.head()

## 4. Quelle 2: EEA (European Environment Agency)

Europäische Daten via `airbase` Python-Package – einfacher API-Wrapper

In [ ]:
# EEA-Daten via airbase Package
# pip install airbase

try:
    import airbase
    
    client = airbase.AirbaseClient()
    
    # Nur Deutschland, validierte Daten 2022
    r = client.request(
        source='Verified',   # validierte Jahresdaten
        countries=['DE'],
        poll=['NO2', 'PM10', 'PM2.5', 'O3'],
        year_from=2022,
        year_to=2022
    )
    
    # Herunterladen (Parquet-Format)
    r.download('data/eea_raw')
    
    # Alle Parquet-Dateien einlesen
    import glob
    parquet_files = glob.glob('data/eea_raw/**/*.parquet', recursive=True)
    
    eea_dfs = [pd.read_parquet(f) for f in parquet_files]
    eea_roh = pd.concat(eea_dfs, ignore_index=True)
    
    print(f'✅ EEA: {len(eea_roh):,} Messwerte aus {len(parquet_files)} Dateien')
    print(eea_roh.columns.tolist())

except ImportError:
    print('⚠️  airbase nicht installiert → pip install airbase')
except Exception as e:
    print(f'⚠️  EEA Fehler: {e}')

In [ ]:
def eea_bereinigen(df_roh):
    """Bringt EEA-Rohdaten in unser einheitliches Schema."""
    df = df_roh.copy()
    
    # Spaltennamen vereinheitlichen (EEA-Konvention)
    spalten_mapping = {
        'AirQualityStationEoICode': 'station_id',
        'DatetimeBegin':           'zeitpunkt',
        'Concentration':           'wert',
        'AirPollutant':            'schadstoff_url'
    }
    df = df.rename(columns={k: v for k, v in spalten_mapping.items() if k in df.columns})
    
    # Schadstoff aus URL extrahieren (EEA nutzt URLs als IDs)
    schadstoff_map = {
        '6001': 'SO2', '7': 'O3', '8': 'NO2',
        '5': 'PM10', '6015': 'CO', '391': 'PM2.5'
    }
    if 'schadstoff_url' in df.columns:
        df['schadstoff'] = df['schadstoff_url'].str.extract(r'/(\d+)$')[0].map(schadstoff_map)
    
    # Formatierung
    df['station_id'] = 'EEA_' + df['station_id'].astype(str)
    df['zeitpunkt']  = pd.to_datetime(df['zeitpunkt'], utc=True).dt.tz_localize(None)
    df['quelle']     = 'EEA'
    df['validiert']  = True
    
    # Nur relevante Spalten
    df = df[['station_id', 'zeitpunkt', 'schadstoff', 'wert', 'validiert', 'quelle']]
    
    # Fehlwerte entfernen
    df = df.dropna(subset=['wert', 'schadstoff'])
    df = df[df['wert'] >= 0]  # negative Werte raus
    
    print(f'✅ EEA bereinigt: {len(df):,} valide Messwerte')
    return df


# eea_messdaten = eea_bereinigen(eea_roh)  # aktivieren wenn EEA-Daten geladen

## 5. Daten in PostgreSQL speichern

In [ ]:
def in_datenbank_speichern(df, tabelle, engine, chunk_size=5000):
    """Lädt DataFrame in PostgreSQL-Tabelle (append, keine Duplikate)."""
    if df.empty:
        print(f'⚠️  Keine Daten für {tabelle}')
        return
    
    df.to_sql(
        name=tabelle,
        con=engine,
        if_exists='append',
        index=False,
        chunksize=chunk_size,
        method='multi'
    )
    print(f'✅ {len(df):,} Zeilen in Tabelle "{tabelle}" gespeichert')


# UBA-Daten speichern
if not uba_messdaten.empty:
    in_datenbank_speichern(uba_messdaten, 'messdaten', engine)

# EEA-Daten speichern (wenn geladen)
# if not eea_messdaten.empty:
#     in_datenbank_speichern(eea_messdaten, 'messdaten', engine)

## 6. Daten prüfen & erste SQL-Abfragen

In [ ]:
# Überblick: Wie viele Messwerte pro Schadstoff?
query = """
SELECT 
    schadstoff,
    quelle,
    COUNT(*)                    AS anzahl_messungen,
    ROUND(AVG(wert)::numeric, 2) AS mittelwert,
    ROUND(MAX(wert)::numeric, 2) AS max_wert,
    MIN(zeitpunkt)::date         AS von,
    MAX(zeitpunkt)::date         AS bis
FROM messdaten
GROUP BY schadstoff, quelle
ORDER BY schadstoff, quelle;
"""

df_check = pd.read_sql(query, engine)
print('📊 Datenbank-Überblick:')
df_check

In [ ]:
# Grenzwertüberschreitungen NO2 (Jahresmittel > 40 µg/m³)
query_grenzwerte = """
SELECT 
    station_id,
    EXTRACT(YEAR FROM zeitpunkt)::INT   AS jahr,
    schadstoff,
    ROUND(AVG(wert)::numeric, 2)        AS jahresmittel,
    COUNT(*)                            AS messtage,
    CASE 
        WHEN AVG(wert) > 40 THEN '🔴 Überschreitung'
        WHEN AVG(wert) > 30 THEN '🟡 Erhöht'
        ELSE '🟢 OK'
    END AS bewertung
FROM messdaten
WHERE schadstoff = 'NO2'
GROUP BY station_id, jahr, schadstoff
HAVING COUNT(*) > 100          -- nur Stationen mit genug Daten
ORDER BY jahresmittel DESC
LIMIT 20;
"""

df_grenzwerte = pd.read_sql(query_grenzwerte, engine)
print('🚨 NO₂ Jahresmittel nach Station (Top 20):')
df_grenzwerte

In [ ]:
# Zeitreihe: Monatlicher Durchschnitt für alle Schadstoffe
query_zeitreihe = """
SELECT 
    DATE_TRUNC('month', zeitpunkt)       AS monat,
    schadstoff,
    ROUND(AVG(wert)::numeric, 2)         AS mittelwert,
    COUNT(DISTINCT station_id)           AS anzahl_stationen
FROM messdaten
GROUP BY monat, schadstoff
ORDER BY monat, schadstoff;
"""

df_zeitreihe = pd.read_sql(query_zeitreihe, engine)
print(f'📈 Monatliche Zeitreihe: {len(df_zeitreihe)} Einträge')
df_zeitreihe.head(12)

## 7. Erste Visualisierung mit Plotly

Schneller Check bevor das Dashboard gebaut wird

In [ ]:
import plotly.express as px

# Zeitreihe plotten
if not df_zeitreihe.empty:
    fig = px.line(
        df_zeitreihe,
        x='monat',
        y='mittelwert',
        color='schadstoff',
        title='🌍 Luftqualität Deutschland 2022 – Monatlicher Durchschnitt',
        labels={'mittelwert': 'Konzentration (µg/m³)', 'monat': 'Monat'},
        template='plotly_white'
    )
    
    # WHO-Grenzwert NO2 einzeichnen
    fig.add_hline(
        y=40, 
        line_dash='dash', 
        line_color='red',
        annotation_text='EU-Grenzwert NO₂ (40 µg/m³)'
    )
    
    fig.show()
else:
    print('⚠️  Keine Daten zum Plotten – erst Daten laden (Schritte 3-5)')

## 8. Nächste Schritte

```
Woche 1 ✅  Datenpipeline (dieses Notebook)
Woche 2 →   Feature Engineering + scikit-learn Modell
             - Zeitfeatures: Monat, Wochentag, Saison
             - Korrelationsanalyse mit Gesundheitsdaten (RKI)
             - Klassifikation: niedriges / mittleres / hohes Risiko
Woche 3 →   Plotly Dash Dashboard + Dokumentation
             - Kartenansicht der Stationen
             - Zeitreihen-Explorer
             - Risikoklassifikation nach Region
```

**Hilfreiche Links:**
- UBA Luftdaten Portal: https://www.umweltbundesamt.de/daten/luft/luftdaten
- EEA Download Portal: https://eeadmz1-downloads-webapp.azurewebsites.net/
- airbase Doku: https://airbase.readthedocs.io
- WHO Grenzwerte: https://www.who.int/news/item/22-09-2021-new-who-global-air-quality-guidelines